<a href="https://colab.research.google.com/github/OmegaS48/Internship_NITP/blob/main/Intern_Day13_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)
train=pd.concat([pd.read_csv('SDNFlow_Dataset_Normal.csv'),pd.read_csv('SDNFlow_Dataset_attack.csv')])
train=train.drop(columns=['ip_proto','src_port','dst_port','flow_ID','eth_type','ipv4_src','ipv4_dst','TnBPDstIP','TnPPDstIP','TnBPSrcIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP'])
train.head()

,flow_duration,tot_len_pkts,tot_pkts,flow_byts_s,flow_pkts_s,pkt_size_average,fwd_header_len,Byts_max_interval_2s,Byts_min_interval_2s,Byts_mean_interval_2s,Byts_std_interval_2s,Pkts_max_interval_2s,Pkts_mean_interval_2s,Pkts_min_interval_2s,Pkts_std_interval_2s,active_max,active_mean,active_min,active_std,idle_max,idle_mean,idle_min,idle_std,Attack,Category
0,6.472,113472,628,17532.756490,97.033375,180.687898,12560,112694,0,28368.00000,48686.674830,625,157.000000,0,270.202702,2,2.0,2,0.0,5,5.0,5,0.0,0,SSH
1,6.465,24492,370,3788.399072,57.231245,66.194595,7400,24294,0,6123.00000,10491.343150,367,92.500000,0,158.487381,2,2.0,2,0.0,5,5.0,5,0.0,0,SSH
2,4.798,2211,4,460.817007,0.833681,552.750000,32,1180,0,737.00000,524.675773,2,1.333333,0,0.942809,4,4.0,4,0.0,0,0.0,0,0.0,0,ICMP
3,595.270,10827817,25627,18189.757590,43.051052,422.515979,205016,44867,11454,36334.95638,2870.840349,103,85.996644,22,4.407512,596,596.0,596,0.0,0,0.0,0,0.0,0,STREAMING
4,595.251,154551644,114704,259641.132900,192.698542,1347.395418,917632,935885,32828,518629.67790,121204.950600,673,384.912752,29,84.666811,596,596.0,596,0.0,0,0.0,0,0.0,0,STREAMING


In [2]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
train['Category']=le.fit_transform(train['Category'])
train['Category'].value_counts()

,count
Category,
2,164974
10,151456
8,123807
7,74644
9,67288
0,53551
11,17787
12,6282
6,1487


In [3]:
Y=train['Category']
X=train

In [4]:
X=X.drop(columns='Category')
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)
from xgboost import XGBClassifier


xgb=XGBClassifier()


estimators=[('xgb',xgb)]


from sklearn.pipeline import Pipeline


pipe=Pipeline(steps=estimators)


pipe

Pipeline(steps=[('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [5]:
pip install scikit-optimize


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 5.8 MB/s eta 0:00:00


In [6]:
from skopt import BayesSearchCV


from skopt.space import Real,Categorical,Integer


search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}

opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=8)


opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3
)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

opt.fit(X_train, Y_train)

Y_pred_train = opt.predict(X_train)
Y_pred_test = opt.predict(X_test)

print("Training Accuracy:", accuracy_score(Y_train, Y_pred_train))
print("Testing Accuracy:", accuracy_score(Y_test, Y_pred_test))

print("Test Confusion Matrix:")
print(confusion_matrix(Y_test, Y_pred_test))

print("Test Classification Report:")
print(classification_report(Y_test, Y_pred_test))